In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_project_ssm")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")
dbutils.widgets.text("file_transformed", "renovacion_prestamo_transformed")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")
file_transformed = dbutils.widgets.get("file_transformed")

In [0]:
df_renov_prestamo_transformed = spark.table(f"{catalogo}.{esquema_source}.{file_transformed}")

In [0]:
df_gold = df_renov_prestamo_transformed.groupBy(col("FLAG_VENTA")).agg(
    # --- Métricas de Volumen ---
    count(col("FLAG_VENTA")).alias("conteo_total_casos"),
    
    # --- Género (Proporciones) ---
    round(avg(col("SEXO_F")), 2).alias("proporcion_mujeres"),
    round(avg(col("SEXO_M")), 2).alias("proporcion_hombres"),
    
    # --- Edad (Demografía) ---
    round(avg(col("EDAD")), 2).alias("edad_promedio"),
    min(col("EDAD")).alias("edad_minima"),
    max(col("EDAD")).alias("edad_maxima"),
    
    # --- Comportamiento de la Oferta ---
    round(avg(col("Plazo_Renovado")), 2).alias("plazo_promedio_meses"),
    round(avg(col("Meses_oferta")), 2).alias("meses_oferta_promedio"),
    
    # --- Capacidad Financiera e Ingresos ---
    round(avg(col("SUELDO_ESTIMADO_LOG")), 2).alias("promedio_sueldo_log"),
    round(avg(col("Linea_Renovado_LOG")), 2).alias("promedio_linea_ofertada_log"),
    round(avg(col("Promed_6Mdeuda_LOG")), 2).alias("promedio_deuda_6m_log"),
    
    # --- Fidelidad ---
    round(avg(col("ANTIGUEDAD_MES_LOG")), 2).alias("promedio_antiguedad_log")
).orderBy(col("FLAG_VENTA").desc())

In [0]:
df_gold.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.golden_renovacion_prestamo")